# Frontend Payload Viewer

Notebook para visualizar de forma simples e humana o conteúdo de `outputs/frontend_payload.json`.

In [1]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px

pd.options.display.float_format = '{:,.2f}'.format

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PAYLOAD_PATH = BASE_DIR / 'outputs' / 'frontend_payload.json'

assert PAYLOAD_PATH.exists(), f'Missing payload file: {PAYLOAD_PATH}'

with PAYLOAD_PATH.open() as f:
    payload = json.load(f)

payload.keys()

dict_keys(['generated_at', 'portfolio_summary', 'assets', 'correlation_matrix'])

## 1. Portfolio summary

In [2]:
summary_df = pd.DataFrame([payload['portfolio_summary']])
summary_df

,asset_count,top_opportunities,highest_risk_assets,diversification_candidates,average_opportunity_score,average_risk_score
0,11,"[GOOGL, AAPL, SPY]","[EH, UDMY, ETH-USD]","[CDR.WA, EH, UDMY]",47.10,46.60


## 2. Assets table

In [3]:
asset_rows = []

for asset in payload['assets']:
    intelligence = asset['intelligence']
    recommendation = asset['recommendation']
    projection = asset['projection']
    asset_rows.append({
        'Ticker': asset['ticker'],
        'Rank': intelligence['asset_rank'],
        'Current Price': intelligence['current_price'],
        'Return 30D %': intelligence['total_return']['30d'],
        'Return 1Y %': intelligence['total_return']['1y'],
        'Return 5Y %': intelligence['total_return']['5y'],
        'Volatility %': intelligence['annualized_volatility_pct'],
        'Max Drawdown %': intelligence['max_drawdown_pct'],
        'Sharpe': intelligence['sharpe_ratio'],
        'Correlation SPY': intelligence['correlation_with_spy'],
        'Risk Score': intelligence['risk_score'],
        'Opportunity Score': intelligence['opportunity_score'],
        'Action': recommendation['action'],
        'Confidence': recommendation['confidence'],
        'Risk Level': recommendation['risk_level'],
        'Trend Outlook': projection['trend_outlook'],
        'Projection Confidence': projection['projection_confidence'],
        'Projection Mid': projection['projected_range']['mid'],
    })

assets_df = pd.DataFrame(asset_rows).sort_values('Rank').reset_index(drop=True)
assets_df

,Ticker,Rank,Current Price,Return 30D %,Return 1Y %,Return 5Y %,Volatility %,Max Drawdown %,Sharpe,Correlation SPY,Risk Score,Opportunity Score,Action,Confidence,Risk Level,Trend Outlook,Projection Confidence,Projection Mid
0,GOOGL,1,387.66,26.91,134.07,238.86,31.25,-44.32,0.89,0.68,37.80,100.00,buy,high,medium,bullish,medium,443.98
1,AAPL,2,298.97,18.05,42.08,140.92,27.44,-33.36,0.70,0.77,33.80,71.10,buy,high,low,bullish,medium,329.82
2,SPY,3,733.73,11.30,24.90,89.17,17.06,-24.50,0.80,1.00,33.30,62.20,hold,high,low,bullish,high,776.40
3,AMZN,4,259.34,21.32,26.14,59.71,35.43,-56.15,0.28,0.72,47.10,57.30,hold,medium,medium,bullish,medium,283.79
4,NXE,5,10.53,-7.31,92.86,130.92,57.82,-54.28,0.32,0.48,48.10,54.40,hold,medium,medium,neutral,low,10.18
5,MSFT,6,417.42,12.12,-7.58,76.08,26.38,-37.15,0.46,0.72,33.30,51.30,hold,medium,low,neutral,medium,426.20
6,CDR.WA,7,257.30,5.75,11.69,55.26,41.52,-62.32,0.22,0.14,30.80,44.20,hold,medium,low,neutral,medium,266.82
7,BTC-USD,8,"77,508.46",2.16,-31.98,90.05,45.30,-76.63,0.20,0.43,50.80,38.40,hold,low,medium,neutral,low,"80,725.07"
8,ETH-USD,9,"2,129.77",-8.01,-51.03,-23.51,60.82,-79.35,-0.06,0.45,60.60,16.00,hold,low,medium,neutral,low,"2,104.92"
9,UDMY,10,4.63,6.19,-31.56,-83.16,59.53,-86.33,-0.55,0.40,61.70,14.50,hold,low,medium,neutral,low,4.55


## 3. Recommendation view

In [4]:
recommendation_df = pd.DataFrame([
    {
        'Ticker': asset['ticker'],
        'Action': asset['recommendation']['action'],
        'Confidence': asset['recommendation']['confidence'],
        'Risk Level': asset['recommendation']['risk_level'],
        'Rationale': asset['recommendation']['rationale'],
        'Key Drivers': ', '.join(asset['recommendation']['key_drivers']),
    }
    for asset in payload['assets']
]).sort_values(['Action', 'Ticker']).reset_index(drop=True)

recommendation_df

,Ticker,Action,Confidence,Risk Level,Rationale,Key Drivers
0,AAPL,buy,high,low,AAPL shows low risk with an opportunity score ...,"positive recent momentum, strong risk-adjusted..."
1,GOOGL,buy,high,medium,GOOGL shows medium risk with an opportunity sc...,"positive recent momentum, strong risk-adjusted..."
2,AMZN,hold,medium,medium,AMZN shows medium risk with an opportunity sco...,"positive recent momentum, deep historical draw..."
3,BTC-USD,hold,low,medium,BTC-USD shows medium risk with an opportunity ...,"positive recent momentum, elevated volatility,..."
4,CDR.WA,hold,medium,low,CDR.WA shows low risk with an opportunity scor...,"positive recent momentum, deep historical draw..."
5,ETH-USD,hold,low,medium,ETH-USD shows medium risk with an opportunity ...,"elevated volatility, deep historical drawdown"
6,MSFT,hold,medium,low,MSFT shows low risk with an opportunity score ...,positive recent momentum
7,NXE,hold,medium,medium,NXE shows medium risk with an opportunity scor...,"elevated volatility, deep historical drawdown"
8,SPY,hold,high,low,SPY shows low risk with an opportunity score o...,"positive recent momentum, strong risk-adjusted..."
9,UDMY,hold,low,medium,UDMY shows medium risk with an opportunity sco...,"positive recent momentum, elevated volatility,..."


## 4. Projection view

In [5]:
projection_df = pd.DataFrame([
    {
        'Ticker': asset['ticker'],
        'Trend Outlook': asset['projection']['trend_outlook'],
        'Projection Confidence': asset['projection']['projection_confidence'],
        'Horizon Days': asset['projection']['projection_horizon_days'],
        'Projected Low': asset['projection']['projected_range']['low'],
        'Projected Mid': asset['projection']['projected_range']['mid'],
        'Projected High': asset['projection']['projected_range']['high'],
        'Scenario Note': asset['projection']['scenario_note'],
    }
    for asset in payload['assets']
]).sort_values('Ticker').reset_index(drop=True)

projection_df

,Ticker,Trend Outlook,Projection Confidence,Horizon Days,Projected Low,Projected Mid,Projected High,Scenario Note
0,AAPL,bullish,medium,90,301.11,329.82,358.54,Bullish short-term scenario driven by recent m...
1,AMZN,bullish,medium,90,251.62,283.79,315.95,Bullish short-term scenario driven by recent m...
2,BTC-USD,neutral,low,90,"68,435.15","80,725.07","93,014.99",Neutral short-term scenario driven by recent m...
3,CDR.WA,neutral,medium,90,229.43,266.82,304.21,Neutral short-term scenario driven by recent m...
4,EH,bearish,low,90,5.50,8.30,11.09,Bearish short-term scenario driven by recent m...
5,ETH-USD,neutral,low,90,"1,651.54","2,104.92","2,558.31",Neutral short-term scenario driven by recent m...
6,GOOGL,bullish,medium,90,401.58,443.98,486.37,Bullish short-term scenario driven by recent m...
7,MSFT,neutral,medium,90,387.66,426.20,464.74,Neutral short-term scenario driven by recent m...
8,NXE,neutral,low,90,8.05,10.18,12.31,Neutral short-term scenario driven by recent m...
9,SPY,bullish,high,90,732.60,776.40,820.20,Bullish short-term scenario driven by recent m...


## 5. Visual ranking

In [6]:
fig = px.bar(
    assets_df,
    x='Ticker',
    y='Opportunity Score',
    color='Action',
    hover_data=['Risk Score', 'Return 5Y %', 'Trend Outlook'],
    title='Opportunity score by asset',
    template='plotly_white',
)
fig.show()

## 6. Risk vs return

In [7]:
fig = px.scatter(
    assets_df,
    x='Volatility %',
    y='Return 5Y %',
    color='Action',
    size='Opportunity Score',
    text='Ticker',
    hover_data=['Risk Score', 'Sharpe', 'Trend Outlook'],
    title='Risk vs 5Y return from the payload',
    template='plotly_white',
)
fig.update_traces(textposition='top center')
fig.show()

## 7. Correlation matrix

In [8]:
correlation_df = pd.DataFrame(payload['correlation_matrix']).sort_index()

fig = px.imshow(
    correlation_df,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlation matrix from the payload',
)
fig.update_layout(height=750, template='plotly_white')
fig.show()